<a href="https://colab.research.google.com/github/Manar-Abdelawal/Spotting-Fake-News/blob/main/Spotting_the_Fake.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 1. Importing Required Libraries
In this step, we import the necessary Python libraries for data manipulation (`pandas`, `numpy`), natural language processing (`nltk`, `re`, `gensim`), machine learning modeling (`scikit-learn`), and data visualization (`matplotlib`, `seaborn`).

In [ ]:
!pip install -q gensim


In [ ]:
import re
import string
import warnings
from gensim.models import Word2Vec
import matplotlib.pyplot as plt
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB, MultinomialNB
from sklearn.svm import LinearSVC

warnings.filterwarnings('ignore')
nltk.download('stopwords', quiet=True)

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

## 2. Data Loading & Labeling
We load the two separate datasets (`True.csv` and `Fake.csv`), assign binary target labels (`1` for True news and `0` for Fake news), and merge them into a single unified DataFrame for processing.

In [ ]:
# Load datasets (update path if loading from Google Drive)
true_df = pd.read_csv('/content/drive/MyDrive/News_dataset/True.csv')
fake_df = pd.read_csv('/content/drive/MyDrive/News_dataset/Fake.csv')

# Assign binary labels
true_df['label'] = 1
fake_df['label'] = 0

# Merge both datasets
df = pd.concat([true_df, fake_df], axis=0).reset_index(drop=True)
print('Datasets loaded and merged successfully!')

## 3. Exploratory Data Analysis (EDA) & Data Cleaning
Here, we inspect the dataset dimensions, check for missing values, remove duplicate news articles to prevent data leakage between training and testing sets, and shuffle the dataset randomly.

In [ ]:
print('--- Dataset Info Before Cleaning ---')
print('Initial Shape:', df.shape)
print('Missing Values:\n', df.isnull().sum())

# Drop missing values and duplicate articles
df.dropna(inplace=True)
duplicates = df.duplicated(subset=['title', 'text']).sum()
print(f'\nRemoved Duplicate Articles: {duplicates}')
df.drop_duplicates(subset=['title', 'text'], inplace=True)

# Shuffle the dataset
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

print('\n--- Dataset Info After Cleaning ---')
print('Final Shape:', df.shape)
print('Class Distribution:\n', df['label'].value_counts())

## 4. NLP Preprocessing & Feature Engineering
We combine the `title` and `text` columns to capture full context, then apply a custom text-cleaning pipeline:
- **Preventing Data Leakage:** Removing the `(Reuters) -` source tag from true news articles so the model learns linguistic patterns rather than memorizing agency names.
- **Feature Engineering:** Replacing URLs (`urllink`), exclamation marks (`exclammark`), and question marks (`questionmark`) with dedicated tokens to preserve stylistic cues of misinformation.
- **NLP Normalization:** Applying lowercasing, punctuation/number removal, tokenization, stopword removal, and Porter Stemming.

In [ ]:
df['content'] = df['title'] + ' ' + df['text']

stop_words = set(stopwords.words('english'))
stemmer = PorterStemmer()


def advanced_nlp_clean(text):
    text = str(text).lower()

    # Remove Reuters source tag to prevent data leakage
    text = re.sub(r'^.*?\(reuters\)\s*-\s*', '', text)

    # Feature Engineering: Convert URLs and emotional punctuation into tokens
    text = re.sub(r'https?://\S+|www\.\S+', ' urllink ', text)
    text = re.sub(r'!+', ' exclammark ', text)
    text = re.sub(r'\?+', ' questionmark ', text)

    # Remove remaining punctuation and numbers
    text = re.sub(r'[%s]' % re.escape(string.punctuation), ' ', text)
    text = re.sub(r'\d+', '', text)

    # Tokenization, Stopwords Removal, and Stemming
    tokens = text.split()
    cleaned_tokens = [
        stemmer.stem(word)
        for word in tokens
        if word not in stop_words and len(word) > 2
    ]

    return ' '.join(cleaned_tokens)


print('Applying NLP preprocessing...')
df['clean_content'] = df['content'].apply(advanced_nlp_clean)
df = df[df['clean_content'] != ''].reset_index(drop=True)
print('Text preprocessing completed!')

## 5. Train-Test Split
We split the cleaned dataset into **80% training data** and **20% testing data** using stratified sampling (`stratify=y`) to maintain the exact class balance across both sets.

In [ ]:
X = df['clean_content']
y = df['label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(
    f'Training samples: {X_train.shape[0]} | Testing samples:'
    f' {X_test.shape[0]}'
)

## 6. Text Vectorization
Since machine learning models require numerical input, we convert the cleaned text into numerical feature vectors using three different techniques:
1. **Count Vectorizer:** Term-frequency bag-of-words representation.
2. **TF-IDF Vectorizer:** Weighs words based on their importance across documents.
3. **Word2Vec Vectorizer:** Captures semantic word embeddings trained on the training corpus, averaged per article.

In [ ]:
print('Initializing Text Vectorizers...')

# A) Count Vectorizer
count_vec = CountVectorizer(max_features=5000)
X_train_count = count_vec.fit_transform(X_train)
X_test_count = count_vec.transform(X_test)

# B) TF-IDF Vectorizer
tfidf_vec = TfidfVectorizer(max_features=5000)
X_train_tfidf = tfidf_vec.fit_transform(X_train)
X_test_tfidf = tfidf_vec.transform(X_test)

# C) Word2Vec Vectorizer
X_train_tokens = [text.split() for text in X_train]
X_test_tokens = [text.split() for text in X_test]

w2v_model = Word2Vec(
    sentences=X_train_tokens, vector_size=100, window=5, min_count=2, workers=4
)


def get_doc_vector(tokens, model, vector_size=100):
    valid_words = [word for word in tokens if word in model.wv]
    if not valid_words:
        return np.zeros(vector_size)
    return np.mean(model.wv[valid_words], axis=0)


X_train_w2v = np.array(
    [get_doc_vector(tokens, w2v_model) for tokens in X_train_tokens]
)
X_test_w2v = np.array(
    [get_doc_vector(tokens, w2v_model) for tokens in X_test_tokens]
)

print('All vectorizers are ready!')

## 7. Model Training & Evaluation
We train and evaluate four classification algorithms (**Logistic Regression**, **Naive Bayes**, **Support Vector Machine**, and **Random Forest**) across all three text vectorization methods, measuring performance via **Accuracy** and **F1-Score**.

In [ ]:
results = []


def evaluate_model(model, X_tr, X_te, y_tr, y_te, model_name, vec_name):
    model.fit(X_tr, y_tr)
    y_pred = model.predict(X_te)
    acc = accuracy_score(y_te, y_pred)
    f1 = f1_score(y_te, y_pred)
    results.append({
        'Model': model_name,
        'Vectorizer': vec_name,
        'Accuracy': round(acc, 4),
        'F1-Score': round(f1, 4),
    })
    print(f'[{model_name} + {vec_name}] -> Accuracy: {acc:.4f} | F1: {f1:.4f}')
    return y_pred


print('--- Training Classification Models ---')

# 1. Logistic Regression
evaluate_model(
    LogisticRegression(max_iter=1000),
    X_train_count,    X_test_count,
    y_train,    y_test,
    'Logistic Regression',
    'Count',
)
evaluate_model(
    LogisticRegression(max_iter=1000),
    X_train_tfidf,    X_test_tfidf,
    y_train,    y_test,
    'Logistic Regression',
    'TF-IDF',
)
evaluate_model(
    LogisticRegression(max_iter=1000),
    X_train_w2v,    X_test_w2v,
    y_train,    y_test,
    'Logistic Regression',
    'Word2Vec',
)

# 2. Naive Bayes
evaluate_model(
    MultinomialNB(),
    X_train_count,    X_test_count,
    y_train,    y_test,
    'Naive Bayes',
    'Count',
)
evaluate_model(
    MultinomialNB(),
    X_train_tfidf,    X_test_tfidf,
    y_train,    y_test,
    'Naive Bayes',
    'TF-IDF',
)
evaluate_model(
    GaussianNB(),
    X_train_w2v,    X_test_w2v,
    y_train,    y_test,
    'Naive Bayes',
    'Word2Vec',
)

# 3. Support Vector Machine (SVM)
evaluate_model(
    LinearSVC(random_state=42),
    X_train_count,    X_test_count,
    y_train,    y_test,
    'SVM',
    'Count',
)
evaluate_model(
    LinearSVC(random_state=42),
    X_train_tfidf,    X_test_tfidf,
    y_train,    y_test,
    'SVM',
    'TF-IDF',
)
evaluate_model(
    LinearSVC(random_state=42),
    X_train_w2v,    X_test_w2v,
    y_train,    y_test,
    'SVM',
    'Word2Vec',
)

# 4. Random Forest (Baseline)
evaluate_model(
    RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
    X_train_count,    X_test_count,
    y_train,    y_test,
    'Random Forest',
    'Count',
)
evaluate_model(
    RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
    X_train_tfidf,    X_test_tfidf,
    y_train,    y_test,
    'Random Forest',
    'TF-IDF',
)
evaluate_model(
    RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
    X_train_w2v,    X_test_w2v,
    y_train,    y_test,
    'Random Forest',
    'Word2Vec',
)

## 7.1 Confusion Matrix Analysis
After training our baseline models, we generate the **Confusion Matrix** for our top-performing model (`SVM + TF-IDF`) to inspect True Positives (TP), True Negatives (TN), False Positives (FP), and False Negatives (FN).

In [ ]:
# Get predictions from the best performing model (SVM + TF-IDF)
best_svm = LinearSVC(random_state=42)
best_svm.fit(X_train_tfidf, y_train)
y_pred_best = best_svm.predict(X_test_tfidf)

# Compute Confusion Matrix
cm = confusion_matrix(y_test, y_pred_best)

print('Confusion Matrix:\n', cm)
print(f'True Negatives (Fake predicted as Fake): {cm[0, 0]}')
print(f'False Positives (Fake predicted as True): {cm[0, 1]}')
print(f'False Negatives (True predicted as Fake): {cm[1, 0]}')
print(f'True Positives (True predicted as True): {cm[1, 1]}')

# Plot Confusion Matrix Heatmap
plt.figure(figsize=(6, 4))
sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='YlGnBu',
    xticklabels=['Predicted Fake (0)', 'Predicted True (1)'],
    yticklabels=['Actual Fake (0)', 'Actual True (1)'],
)
plt.title(
    'Confusion Matrix - Best Model (SVM + TF-IDF)',
    fontsize=12,
    fontweight='bold',
)
plt.tight_layout()
plt.show()

## 8. Refining Random Forest with TF-IDF
To further improve the Random Forest classifier, we expand the TF-IDF representation to include bigrams (`ngram_range=(1, 2)`) with `10,000` features and tune key tree hyperparameters (`n_estimators=200`, `max_depth=50`, `min_samples_split=5`).

In [ ]:
print('--- Refining Random Forest + TF-IDF ---')

tfidf_refined = TfidfVectorizer(max_features=10000, ngram_range=(1, 2))
X_train_tfidf_ref = tfidf_refined.fit_transform(X_train)
X_test_tfidf_ref = tfidf_refined.transform(X_test)

rf_refined = RandomForestClassifier(
    n_estimators=200,
    max_depth=50,
    min_samples_split=5,
    random_state=42,
    n_jobs=-1,
)
y_pred_rf_ref = evaluate_model(
    rf_refined,
    X_train_tfidf_ref,
    X_test_tfidf_ref,
    y_train,
    y_test,
    'Random Forest (Refined)',
    'TF-IDF (1,2-gram)',
)

## 9. Final Comparison & Visualization
Finally, we rank all models by their **F1-Score**, plot a comparative **Bar Chart** across all models and vectorizers.

In [ ]:
# Summary Table
results_df = pd.DataFrame(results).sort_values(by='F1-Score', ascending=False)
print('=== Summary of All Models ===')
print(results_df.to_string(index=False))

# Create a combined label for plotting
results_df['Model_Vec'] = (
    results_df['Model'] + ' (' + results_df['Vectorizer'] + ')'
)

# 1. Bar Chart comparing F1-Scores
plt.figure(figsize=(12, 6))
ax = sns.barplot(
    data=results_df, x='F1-Score', y='Model_Vec', palette='viridis'
)
plt.title(
    'Model Performance Comparison (F1-Score)', fontsize=14, fontweight='bold'
)
plt.xlabel('F1-Score', fontsize=12)
plt.ylabel('Model & Vectorizer', fontsize=12)
plt.xlim(0.88, 1.0)  # Zoom in to clearly show differences

# Add exact score labels on bars
for container in ax.containers:
    ax.bar_label(container, fmt='%.4f', padding=4, fontsize=10)

plt.tight_layout()
plt.show()


In [ ]:
results_df.drop(columns=['Model_Vec'])

In [ ]:
from IPython.display import HTML, display

samples = [
    "SHOCKING!! Secret aliens found controlling the government, click here http://secret-truth.com!!!",
    "The central bank announced on Thursday that interest rates will remain unchanged following its monthly policy meeting."
]

rows_html = ""
for text in samples:
    cleaned = advanced_nlp_clean(text)
    pred = best_svm.predict(tfidf_vec.transform([cleaned]))[0]
    badge = (
        '<span style="color: #16a34a; font-weight: bold;">True News ✅</span>'
        if pred == 1 else
        '<span style="color: #dc2626; font-weight: bold;">Fake News ❌</span>'
    )
    rows_html += f"""
    <tr style="border-bottom: 1px solid #e5e7eb;">
        <td style="padding: 12px 16px; text-align: left; color: #1f2937;">{text}</td>
        <td style="padding: 12px 16px; text-align: left; white-space: nowrap;">{badge}</td>
    </tr>
    """

table_html = f"""
<table style="width: 100%; border-collapse: collapse; font-family: Arial, sans-serif; font-size: 14px; background-color: #ffffff; border: 1px solid #e5e7eb; border-radius: 8px; overflow: hidden;">
    <thead>
        <tr style="background-color: #f3f4f6; border-bottom: 2px solid #d1d5db;">
            <th style="padding: 12px 16px; text-align: left; color: #111827; font-weight: bold;">Sample News Input</th>
            <th style="padding: 12px 16px; text-align: left; color: #111827; font-weight: bold;">Model Prediction</th>
        </tr>
    </thead>
    <tbody>
        {rows_html}
    </tbody>
</table>
"""

display(HTML(table_html))